<a href="https://colab.research.google.com/github/barr92/public-house-price-data/blob/main/UK%20Land%20Registry%20Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Signals BI — Price Paid Aggregation Pipeline
**Project:** `signalsbi-new` | **Dataset:** `uk_sold_prices` (prod) / `uk_sold_prices_dev` (dev)

## How to use
| Scenario | Settings |
|---|---|
| First ever run | `ENV = 'dev'`, `FIRST_RUN = True` |
| Validate dev, promote to prod | Run Cell 6 with `PROMOTE = True` |
| Every month after | `ENV = 'prod'`, `FIRST_RUN = False` |

**You never need to manually download any files.**

## Tables produced
| Table | Description |
|---|---|
| `transactions_raw` | Source of truth — one row per sale, all history |
| `monthly_trends_sector` | Monthly volume + prices per sector |
| `monthly_trends_sector_by_type` | Monthly volume + prices per sector x property type |
| `new_dev_trends_sector` | New build vs existing monthly prices + MoM/YoY diffs |
| `sales_trends_sector` | Monthly sales counts per sector |
| `price_growth_sector` | 3m / 6m / 12m / 5yr / 10yr growth |
| `annual_price_growth_sector` | Annual prices all years, new build vs existing |
| `cagr_sector_by_type` | 3yr / 5yr / 10yr CAGR per sector x property type |
| `sector_percentile_bands` | P10-P90 + national / district rank |
| `price_growth_local_authority` | LA-level growth + top 20 ranks |
| `street_summary` | Street-level new build vs existing |

## 0. Install dependencies

In [1]:
# !pip install google-cloud-bigquery google-cloud-bigquery-storage pandas db-dtypes requests --quiet

import io
import time
import requests
import pandas as pd
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from google.cloud import bigquery

print('Imports OK')

Imports OK


## 1. Configuration
**This is the only cell you need to change between runs.**

In [8]:
# ── CHANGE THESE TWO SETTINGS EACH RUN ─────────────────────────────────────
ENV       = 'dev'   # 'dev' or 'prod'
FIRST_RUN = True    # True  = full history backfill 1995->present + create tables
                    # False = latest month only, merge into existing tables
# ────────────────────────────────────────────────────────────────────────────

PROJECT  = 'signalsbi-new'
DATASETS = {'prod': 'uk_sold_prices', 'dev': 'uk_sold_prices_dev'}
DATASET       = DATASETS[ENV]
D             = f'`{PROJECT}.{DATASET}`'
RAW_TABLE     = f'{PROJECT}.{DATASET}.transactions_raw'
STAGING_TABLE = f'{PROJECT}.{DATASET}.transactions_staging'

S3_BASE = 'https://price-paid-data.publicdata.landregistry.gov.uk'
MONTHLY_URL  = f'{S3_BASE}/pp-monthly-update-new-version.csv'
HISTORY_FROM = 1995

def yearly_url(yr): return f'{S3_BASE}/pp-{yr}.csv'

def get_latest_complete_month():
    today = date.today()
    return (today.replace(day=1) - relativedelta(months=1)
            if today.day >= 25
            else today.replace(day=1) - relativedelta(months=2))

latest_month = get_latest_complete_month()
date_to   = latest_month + relativedelta(months=1) - relativedelta(days=1)
date_from = latest_month - relativedelta(years=31)

print(f'Environment  : {ENV.upper()}')
print(f'First run    : {FIRST_RUN}')
print(f'Dataset      : {PROJECT}.{DATASET}')
print(f'Latest month : {latest_month.strftime("%B %Y")}')
print(f'Window       : {date_from} -> {date_to}')

Environment  : DEV
First run    : True
Dataset      : signalsbi-new.uk_sold_prices_dev
Latest month : April 2026
Window       : 1995-04-01 -> 2026-04-30


## 2. Create dataset and `transactions_raw` if they don't exist

In [5]:
from google.colab import auth
auth.authenticate_user()

In [6]:
client = bigquery.Client(project=PROJECT)

# Create dataset (no-op if already exists)
ds = bigquery.Dataset(f'{PROJECT}.{DATASET}')
ds.location = 'EU'
client.create_dataset(ds, exists_ok=True)
print(f'Dataset `{PROJECT}.{DATASET}` ready.')

# Create transactions_raw with schema, partitioning, clustering
client.query(f'''
CREATE TABLE IF NOT EXISTS `{RAW_TABLE}`
(
  transaction_id      STRING    NOT NULL,
  transaction_date    DATE,
  price               INT64,
  postcode            STRING,
  postcode_sector     STRING,
  postcode_district   STRING,
  property_type       STRING,
  tenure              STRING,
  new_build           BOOL,
  paon                STRING,
  saon                STRING,
  street              STRING,
  locality            STRING,
  town                STRING,
  local_authority     STRING,
  county              STRING,
  year_month          STRING,
  year                INT64,
  record_status       STRING,
  ingested_at         TIMESTAMP
)
PARTITION BY transaction_date
CLUSTER BY postcode_sector, property_type
OPTIONS (require_partition_filter = false)
''').result()
print(f'Table `{RAW_TABLE}` ready.')

Dataset `signalsbi-new.uk_sold_prices_dev` ready.
Table `signalsbi-new.uk_sold_prices_dev.transactions_raw` ready.


## 3. Download & ingest HMLR CSV(s)
- **FIRST_RUN = True** : loops 1995 -> current year (yearly files) then pulls monthly update. One-time only.
- **FIRST_RUN = False** : downloads monthly update only (~20 MB). Run every month.

In [ ]:
COL_NAMES = [
    'transaction_id','price','transaction_date','postcode',
    'property_type','new_build','tenure',
    'paon','saon','street','locality','town',
    'local_authority','county','transaction_category','record_status'
]
PROP_MAP   = {'D':'Detached','S':'Semi-Detached','T':'Terraced','F':'Flat','O':'Other'}
TENURE_MAP = {'F':'Freehold','L':'Leasehold'}

MERGE_SQL = f'''
MERGE `{RAW_TABLE}` AS target
USING `{STAGING_TABLE}` AS source ON target.transaction_id = source.transaction_id
WHEN MATCHED AND source.record_status = 'D' THEN DELETE
WHEN MATCHED AND source.record_status IN ('A','C') THEN UPDATE SET
  transaction_date=source.transaction_date,price=source.price,
  postcode=source.postcode,postcode_sector=source.postcode_sector,
  postcode_district=source.postcode_district,property_type=source.property_type,
  tenure=source.tenure,new_build=source.new_build,
  paon=source.paon,saon=source.saon,street=source.street,
  locality=source.locality,town=source.town,
  local_authority=source.local_authority,county=source.county,
  year_month=source.year_month,year=source.year,
  record_status=source.record_status,ingested_at=source.ingested_at
WHEN NOT MATCHED AND source.record_status != 'D' THEN INSERT (
  transaction_id,transaction_date,price,postcode,postcode_sector,postcode_district,
  property_type,tenure,new_build,paon,saon,street,locality,town,
  local_authority,county,year_month,year,record_status,ingested_at
) VALUES (
  source.transaction_id,source.transaction_date,source.price,
  source.postcode,source.postcode_sector,source.postcode_district,
  source.property_type,source.tenure,source.new_build,
  source.paon,source.saon,source.street,source.locality,source.town,
  source.local_authority,source.county,source.year_month,source.year,
  source.record_status,source.ingested_at
)
'''

def download_csv(url):
    print(f'  Downloading {url} ...', end=' ', flush=True)
    resp = requests.get(url, timeout=600, stream=True)
    resp.raise_for_status()
    chunks, mb = [], 0
    for chunk in resp.iter_content(1024*1024):
        chunks.append(chunk); mb += len(chunk)/1024/1024
    print(f'{mb:.0f} MB')
    return b''.join(chunks)

def parse_and_clean(raw_bytes):
    df = pd.read_csv(io.BytesIO(raw_bytes), header=None, names=COL_NAMES, dtype=str, encoding='utf-8')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce').dt.date
    df['price']            = pd.to_numeric(df['price'], errors='coerce').astype('Int64')
    df['new_build']        = df['new_build'].map({'Y':True,'N':False})
    df['property_type']    = df['property_type'].str.strip().map(PROP_MAP)
    df['tenure']           = df['tenure'].str.strip().map(TENURE_MAP)
    df['record_status']    = df['record_status'].str.strip()
    df['postcode']         = df['postcode'].str.strip().str.upper()
    df = df.dropna(subset=['transaction_id','transaction_date','price','postcode'])
    df['postcode_sector']   = df['postcode'].str.rsplit(' ',n=1).str[0] + ' ' + df['postcode'].str[-3]
    df['postcode_district'] = df['postcode'].str.split(' ').str[0]
    df['year_month']        = df['transaction_date'].apply(lambda d: d.strftime('%Y-%m') if d else None)
    df['year']              = df['transaction_date'].apply(lambda d: d.year if d else None)
    df['ingested_at']       = datetime.utcnow()
    cols = ['transaction_id','transaction_date','price','postcode','postcode_sector',
            'postcode_district','property_type','tenure','new_build','paon','saon',
            'street','locality','town','local_authority','county',
            'year_month','year','record_status','ingested_at']
    return df[[c for c in cols if c in df.columns]]

def upsert(df, label):
    before = len(df)
    df = df.drop_duplicates(subset=['transaction_id'], keep='last')
    dupes = before - len(df)
    if dupes > 0:
        print(f'  Removed {dupes:,} duplicate transaction_ids')
    print(f'  Upserting {len(df):,} rows ({label})...')
    client.load_table_from_dataframe(
        df, STAGING_TABLE,
        job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE', autodetect=True)
    ).result()
    client.query(MERGE_SQL).result()
    print(f'  Merge complete ({label})')

# ── Main ingest ───────────────────────────────────────────────────────────
if FIRST_RUN:
    print('FIRST RUN — downloading full history 1995 -> present...')
    for yr in range(HISTORY_FROM, date.today().year):
        try:
            upsert(parse_and_clean(download_csv(yearly_url(yr))), str(yr))
            time.sleep(1)
        except Exception as e:
            print(f'  WARNING: {yr} failed — {e}. Skipping.')
    print('Downloading current-year monthly update...')
    upsert(parse_and_clean(download_csv(MONTHLY_URL)), 'monthly update')
else:
    print('MONTHLY UPDATE — downloading latest month only...')
    upsert(parse_and_clean(download_csv(MONTHLY_URL)), 'monthly update')

n = client.query(f'SELECT COUNT(*) AS n FROM `{RAW_TABLE}`').to_dataframe().iloc[0]['n']
print(f'\ntransactions_raw row count: {n:,}')

FIRST RUN — downloading full history 1995 -> present...


/tmp/ipykernel_4454/2063010167.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df['ingested_at']       = datetime.utcnow()


  Upserting 796,589 rows (1995)...

Location: EU
Job ID: e3198812-1c42-4d0e-8101-cab2c4922045
. Skipping.


/tmp/ipykernel_4454/2063010167.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df['ingested_at']       = datetime.utcnow()


  Upserting 964,802 rows (1996)...

Location: EU
Job ID: e2ecf0b5-ec30-4d85-9d8e-145bd31aae35
. Skipping.


...

  Upserting 1,093,922 rows (1997)...

Location: EU
Job ID: 9fbc1f95-aaa1-4c70-938c-9a0f403bf6bd
. Skipping.


/tmp/ipykernel_4454/2063010167.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df['ingested_at']       = datetime.utcnow()


  Upserting 1,050,000 rows (1998)...

Location: EU
Job ID: 88f94bc5-4c87-4089-8f6f-358962125749
. Skipping.


/tmp/ipykernel_4454/2063010167.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df['ingested_at']       = datetime.utcnow()


  Upserting 1,194,351 rows (1999)...

Location: EU
Job ID: 0c3f5a06-ab03-4088-860c-e4e7ebb6bd9d
. Skipping.


## 4. Refresh aggregated tables
Runs in dependency order. Each query fully replaces its target table.

In [ ]:
def run_query(name, sql):
    print(f'  Refreshing {name}...', end=' ', flush=True)
    client.query(sql).result()
    print('done')

### 4a. Monthly trends — sector

In [ ]:
run_query('monthly_trends_sector', f'''
CREATE OR REPLACE TABLE {D}.monthly_trends_sector AS
SELECT postcode_sector, year_month,
  COUNT(*)                                  AS transaction_count,
  APPROX_QUANTILES(price,100)[OFFSET(50)]   AS median_price,
  ROUND(AVG(price),0)                       AS mean_price,
  MIN(price) AS min_price, MAX(price) AS max_price,
  CURRENT_TIMESTAMP()                       AS refreshed_at
FROM {D}.transactions_raw
WHERE record_status != 'D'
GROUP BY 1,2
''')

### 4b. Monthly trends — sector x property type

In [ ]:
run_query('monthly_trends_sector_by_type', f'''
CREATE OR REPLACE TABLE {D}.monthly_trends_sector_by_type AS
SELECT postcode_sector, property_type, year_month,
  COUNT(*)                                  AS transaction_count,
  APPROX_QUANTILES(price,100)[OFFSET(50)]   AS median_price,
  ROUND(AVG(price),0)                       AS mean_price,
  MIN(price) AS min_price, MAX(price) AS max_price,
  CURRENT_TIMESTAMP()                       AS refreshed_at
FROM {D}.transactions_raw
WHERE record_status != 'D'
GROUP BY 1,2,3
''')

### 4c. New development trends — new build vs existing

In [ ]:
run_query('new_dev_trends_sector', f'''
CREATE OR REPLACE TABLE {D}.new_dev_trends_sector AS
WITH monthly AS (
  SELECT postcode_sector, year_month,
    COUNTIF(new_build=TRUE)                                        AS new_build_count,
    COUNTIF(new_build=FALSE)                                       AS existing_count,
    COUNT(*)                                                       AS total_count,
    APPROX_QUANTILES(IF(new_build,price,NULL),100)[OFFSET(50)]    AS new_build_median,
    APPROX_QUANTILES(IF(NOT new_build,price,NULL),100)[OFFSET(50)] AS existing_median,
    ROUND(AVG(IF(new_build,price,NULL)),0)                        AS new_build_mean,
    ROUND(AVG(IF(NOT new_build,price,NULL)),0)                    AS existing_mean
  FROM {D}.transactions_raw
  WHERE record_status != 'D' AND new_build IS NOT NULL
  GROUP BY 1,2
)
SELECT *,
  ROUND(SAFE_DIVIDE(new_build_median-existing_median,existing_median)*100,2) AS new_build_premium_pct_median,
  ROUND(SAFE_DIVIDE(new_build_mean-existing_mean,existing_mean)*100,2)       AS new_build_premium_pct_mean,
  ROUND(SAFE_DIVIDE(new_build_median-LAG(new_build_median) OVER (PARTITION BY postcode_sector ORDER BY year_month),
    LAG(new_build_median) OVER (PARTITION BY postcode_sector ORDER BY year_month))*100,2) AS new_build_mom_pct,
  ROUND(SAFE_DIVIDE(existing_median-LAG(existing_median) OVER (PARTITION BY postcode_sector ORDER BY year_month),
    LAG(existing_median) OVER (PARTITION BY postcode_sector ORDER BY year_month))*100,2)  AS existing_mom_pct,
  ROUND(SAFE_DIVIDE(new_build_median-LAG(new_build_median,12) OVER (PARTITION BY postcode_sector ORDER BY year_month),
    LAG(new_build_median,12) OVER (PARTITION BY postcode_sector ORDER BY year_month))*100,2) AS new_build_yoy_pct,
  ROUND(SAFE_DIVIDE(existing_median-LAG(existing_median,12) OVER (PARTITION BY postcode_sector ORDER BY year_month),
    LAG(existing_median,12) OVER (PARTITION BY postcode_sector ORDER BY year_month))*100,2)  AS existing_yoy_pct,
  CURRENT_TIMESTAMP() AS refreshed_at
FROM monthly
''')

### 4d. Sales trends — monthly counts per sector

In [ ]:
run_query('sales_trends_sector', f'''
CREATE OR REPLACE TABLE {D}.sales_trends_sector AS
WITH monthly AS (
  SELECT postcode_sector, year_month,
    COUNT(*)                                 AS total_sales,
    COUNTIF(new_build=TRUE)                  AS new_build_sales,
    COUNTIF(new_build=FALSE)                 AS existing_sales,
    COUNTIF(property_type='Detached')        AS detached_sales,
    COUNTIF(property_type='Semi-Detached')   AS semi_detached_sales,
    COUNTIF(property_type='Terraced')        AS terraced_sales,
    COUNTIF(property_type='Flat')            AS flat_sales,
    COUNTIF(tenure='Freehold')               AS freehold_sales,
    COUNTIF(tenure='Leasehold')              AS leasehold_sales
  FROM {D}.transactions_raw
  WHERE record_status != 'D'
  GROUP BY 1,2
)
SELECT *,
  ROUND(SAFE_DIVIDE(total_sales-LAG(total_sales) OVER (PARTITION BY postcode_sector ORDER BY year_month),
    LAG(total_sales) OVER (PARTITION BY postcode_sector ORDER BY year_month))*100,2) AS total_sales_mom_pct,
  ROUND(SAFE_DIVIDE(total_sales-LAG(total_sales,12) OVER (PARTITION BY postcode_sector ORDER BY year_month),
    LAG(total_sales,12) OVER (PARTITION BY postcode_sector ORDER BY year_month))*100,2) AS total_sales_yoy_pct,
  CURRENT_TIMESTAMP() AS refreshed_at
FROM monthly
''')

### 4e. Price growth — sector (3m / 6m / 12m / 5yr / 10yr)

In [ ]:
run_query('price_growth_sector', f'''
CREATE OR REPLACE TABLE {D}.price_growth_sector AS
WITH latest AS (SELECT MAX(year_month) AS latest_ym FROM {D}.monthly_trends_sector),
anchors AS (
  SELECT 'dynamic' AS anchor_type, PARSE_DATE('%Y-%m',latest_ym) AS anchor_date FROM latest
  UNION ALL
  SELECT 'year_end', PARSE_DATE('%Y-%m',MAX(year_month))
  FROM {D}.monthly_trends_sector WHERE RIGHT(year_month,2)='12'
),
base AS (
  SELECT a.anchor_type,a.anchor_date,t.postcode_sector,
    t.median_price,t.mean_price,t.transaction_count,
    DATE_DIFF(a.anchor_date,PARSE_DATE('%Y-%m',t.year_month),MONTH) AS months_ago
  FROM {D}.monthly_trends_sector t CROSS JOIN anchors a
)
SELECT postcode_sector,anchor_type,anchor_date,
  MAX(IF(months_ago=0,  median_price,NULL)) AS median_price_now,
  MAX(IF(months_ago=0,  mean_price,  NULL)) AS mean_price_now,
  MAX(IF(months_ago=3,  median_price,NULL)) AS median_price_3m_ago,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=3,median_price,NULL)),MAX(IF(months_ago=3,median_price,NULL)))*100,2) AS median_growth_3m,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL))-MAX(IF(months_ago=3,mean_price,NULL)),MAX(IF(months_ago=3,mean_price,NULL)))*100,2) AS mean_growth_3m,
  SUM(IF(months_ago BETWEEN 0 AND 2,transaction_count,0)) AS count_3m,
  MAX(IF(months_ago=6,  median_price,NULL)) AS median_price_6m_ago,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=6,median_price,NULL)),MAX(IF(months_ago=6,median_price,NULL)))*100,2) AS median_growth_6m,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL))-MAX(IF(months_ago=6,mean_price,NULL)),MAX(IF(months_ago=6,mean_price,NULL)))*100,2) AS mean_growth_6m,
  SUM(IF(months_ago BETWEEN 0 AND 5,transaction_count,0)) AS count_6m,
  MAX(IF(months_ago=12, median_price,NULL)) AS median_price_12m_ago,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=12,median_price,NULL)),MAX(IF(months_ago=12,median_price,NULL)))*100,2) AS median_growth_12m,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL))-MAX(IF(months_ago=12,mean_price,NULL)),MAX(IF(months_ago=12,mean_price,NULL)))*100,2) AS mean_growth_12m,
  SUM(IF(months_ago BETWEEN 0 AND 11,transaction_count,0)) AS count_12m,
  MAX(IF(months_ago=60, median_price,NULL)) AS median_price_5yr_ago,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=60,median_price,NULL)),MAX(IF(months_ago=60,median_price,NULL)))*100,2) AS median_growth_5yr,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL))-MAX(IF(months_ago=60,mean_price,NULL)),MAX(IF(months_ago=60,mean_price,NULL)))*100,2) AS mean_growth_5yr,
  SUM(IF(months_ago BETWEEN 0 AND 59,transaction_count,0)) AS count_5yr,
  MAX(IF(months_ago=120,median_price,NULL)) AS median_price_10yr_ago,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=120,median_price,NULL)),MAX(IF(months_ago=120,median_price,NULL)))*100,2) AS median_growth_10yr,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL))-MAX(IF(months_ago=120,mean_price,NULL)),MAX(IF(months_ago=120,mean_price,NULL)))*100,2) AS mean_growth_10yr,
  SUM(IF(months_ago BETWEEN 0 AND 119,transaction_count,0)) AS count_10yr,
  CURRENT_TIMESTAMP() AS refreshed_at
FROM base GROUP BY 1,2,3
''')

### 4f. Annual price growth — all years, new build vs existing (1995->last full year)

In [ ]:
run_query('annual_price_growth_sector', f'''
CREATE OR REPLACE TABLE {D}.annual_price_growth_sector AS
WITH annual AS (
  SELECT postcode_sector, year, property_type, new_build,
    COUNT(*)                                 AS transaction_count,
    APPROX_QUANTILES(price,100)[OFFSET(50)]  AS median_price,
    ROUND(AVG(price),0)                      AS mean_price
  FROM {D}.transactions_raw
  WHERE record_status != 'D' AND new_build IS NOT NULL
    AND year < EXTRACT(YEAR FROM CURRENT_DATE())
  GROUP BY 1,2,3,4
),
annual_all AS (
  SELECT postcode_sector, year, 'All' AS property_type, new_build,
    SUM(transaction_count) AS transaction_count,
    ROUND(SUM(median_price*transaction_count)/SUM(transaction_count),0) AS median_price,
    ROUND(SUM(mean_price*transaction_count)/SUM(transaction_count),0)   AS mean_price
  FROM annual GROUP BY 1,2,3,4
),
combined AS (SELECT * FROM annual UNION ALL SELECT * FROM annual_all)
SELECT *,
  ROUND(SAFE_DIVIDE(median_price-LAG(median_price) OVER w,LAG(median_price) OVER w)*100,2) AS median_yoy_growth_pct,
  ROUND(SAFE_DIVIDE(mean_price-LAG(mean_price) OVER w,LAG(mean_price) OVER w)*100,2)       AS mean_yoy_growth_pct,
  ROUND(median_price/FIRST_VALUE(median_price) OVER w*100,1)                               AS median_index,
  CURRENT_TIMESTAMP() AS refreshed_at
FROM combined
WINDOW w AS (PARTITION BY postcode_sector,property_type,new_build ORDER BY year
             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
ORDER BY postcode_sector,property_type,new_build,year
''')

### 4g. CAGR by property type

In [ ]:
run_query('cagr_sector_by_type', f'''
CREATE OR REPLACE TABLE {D}.cagr_sector_by_type AS
WITH latest AS (SELECT MAX(year_month) AS latest_ym FROM {D}.monthly_trends_sector_by_type),
anchors AS (
  SELECT 'dynamic' AS anchor_type,PARSE_DATE('%Y-%m',latest_ym) AS anchor_date FROM latest
  UNION ALL
  SELECT 'year_end',PARSE_DATE('%Y-%m',MAX(year_month))
  FROM {D}.monthly_trends_sector_by_type WHERE RIGHT(year_month,2)='12'
),
base AS (
  SELECT a.anchor_type,a.anchor_date,t.postcode_sector,t.property_type,
    t.median_price,t.mean_price,
    DATE_DIFF(a.anchor_date,PARSE_DATE('%Y-%m',t.year_month),MONTH) AS months_ago
  FROM {D}.monthly_trends_sector_by_type t CROSS JOIN anchors a
  UNION ALL
  SELECT a.anchor_type,a.anchor_date,t.postcode_sector,'All',
    t.median_price,t.mean_price,
    DATE_DIFF(a.anchor_date,PARSE_DATE('%Y-%m',t.year_month),MONTH) AS months_ago
  FROM {D}.monthly_trends_sector t CROSS JOIN anchors a
)
SELECT postcode_sector,property_type,anchor_type,anchor_date,
  ROUND((POW(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL)),MAX(IF(months_ago=36,median_price,NULL))),1.0/3)-1)*100,2) AS cagr_3yr_median,
  ROUND((POW(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL)),MAX(IF(months_ago=36,mean_price,NULL))),1.0/3)-1)*100,2) AS cagr_3yr_mean,
  ROUND((POW(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL)),MAX(IF(months_ago=60,median_price,NULL))),1.0/5)-1)*100,2) AS cagr_5yr_median,
  ROUND((POW(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL)),MAX(IF(months_ago=60,mean_price,NULL))),1.0/5)-1)*100,2) AS cagr_5yr_mean,
  ROUND((POW(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL)),MAX(IF(months_ago=120,median_price,NULL))),1.0/10)-1)*100,2) AS cagr_10yr_median,
  ROUND((POW(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL)),MAX(IF(months_ago=120,mean_price,NULL))),1.0/10)-1)*100,2) AS cagr_10yr_mean,
  MAX(IF(months_ago=36,median_price,NULL)) AS base_price_3yr_median,
  MAX(IF(months_ago=60,median_price,NULL)) AS base_price_5yr_median,
  MAX(IF(months_ago=120,median_price,NULL)) AS base_price_10yr_median,
  MAX(IF(months_ago=0,median_price,NULL)) AS end_price_median,
  MAX(IF(months_ago=0,mean_price,NULL)) AS end_price_mean,
  CURRENT_TIMESTAMP() AS refreshed_at
FROM base GROUP BY 1,2,3,4
''')

### 4h. Sector percentile bands

In [ ]:
run_query('sector_percentile_bands', f'''
CREATE OR REPLACE TABLE {D}.sector_percentile_bands AS
WITH latest_ym AS (SELECT MAX(year_month) AS ym FROM {D}.transactions_raw),
recent AS (
  SELECT r.postcode_sector,r.postcode_district,r.property_type,r.price
  FROM {D}.transactions_raw r, latest_ym
  WHERE r.year_month=latest_ym.ym AND r.record_status!='D'
),
all_type AS (
  SELECT postcode_sector,postcode_district,property_type,price FROM recent
  UNION ALL SELECT postcode_sector,postcode_district,'All',price FROM recent
),
bands AS (
  SELECT postcode_sector,postcode_district,property_type,
    APPROX_QUANTILES(price,100)[OFFSET(10)] AS p10_price,
    APPROX_QUANTILES(price,100)[OFFSET(25)] AS p25_price,
    APPROX_QUANTILES(price,100)[OFFSET(50)] AS p50_price,
    APPROX_QUANTILES(price,100)[OFFSET(75)] AS p75_price,
    APPROX_QUANTILES(price,100)[OFFSET(90)] AS p90_price
  FROM all_type GROUP BY 1,2,3
)
SELECT b.*,
  ROUND(PERCENT_RANK() OVER (PARTITION BY property_type ORDER BY p50_price)*100,1) AS national_percentile_rank,
  ROUND(PERCENT_RANK() OVER (PARTITION BY postcode_district,property_type ORDER BY p50_price)*100,1) AS district_percentile_rank,
  (SELECT PARSE_DATE('%Y-%m',ym) FROM latest_ym) AS anchor_date,
  CURRENT_TIMESTAMP() AS refreshed_at
FROM bands b
''')

### 4i. Local authority price growth

In [ ]:
run_query('price_growth_local_authority', f'''
CREATE OR REPLACE TABLE {D}.price_growth_local_authority AS
WITH la_monthly AS (
  SELECT local_authority,year_month,
    COUNT(*) AS transaction_count,
    APPROX_QUANTILES(price,100)[OFFSET(50)] AS median_price,
    ROUND(AVG(price),0) AS mean_price
  FROM {D}.transactions_raw
  WHERE record_status!='D' AND local_authority IS NOT NULL
  GROUP BY 1,2
),
latest AS (SELECT MAX(year_month) AS latest_ym FROM la_monthly),
anchors AS (
  SELECT 'dynamic' AS anchor_type,PARSE_DATE('%Y-%m',latest_ym) AS anchor_date FROM latest
  UNION ALL SELECT 'year_end',PARSE_DATE('%Y-%m',MAX(year_month))
  FROM la_monthly WHERE RIGHT(year_month,2)='12'
),
base AS (
  SELECT a.anchor_type,a.anchor_date,t.local_authority,
    t.median_price,t.mean_price,t.transaction_count,
    DATE_DIFF(a.anchor_date,PARSE_DATE('%Y-%m',t.year_month),MONTH) AS months_ago
  FROM la_monthly t CROSS JOIN anchors a
)
SELECT local_authority,anchor_type,anchor_date,
  MAX(IF(months_ago=0,median_price,NULL)) AS median_price_current,
  MAX(IF(months_ago=0,mean_price,NULL))   AS mean_price_current,
  SUM(IF(months_ago BETWEEN 0 AND 11,transaction_count,0)) AS transaction_count_12m,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=3,  median_price,NULL)),MAX(IF(months_ago=3,  median_price,NULL)))*100,2) AS median_growth_3m,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=6,  median_price,NULL)),MAX(IF(months_ago=6,  median_price,NULL)))*100,2) AS median_growth_6m,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=12, median_price,NULL)),MAX(IF(months_ago=12, median_price,NULL)))*100,2) AS median_growth_12m,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=60, median_price,NULL)),MAX(IF(months_ago=60, median_price,NULL)))*100,2) AS median_growth_5yr,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=120,median_price,NULL)),MAX(IF(months_ago=120,median_price,NULL)))*100,2) AS median_growth_10yr,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL))-MAX(IF(months_ago=3,  mean_price,NULL)),MAX(IF(months_ago=3,  mean_price,NULL)))*100,2) AS mean_growth_3m,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL))-MAX(IF(months_ago=6,  mean_price,NULL)),MAX(IF(months_ago=6,  mean_price,NULL)))*100,2) AS mean_growth_6m,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL))-MAX(IF(months_ago=12, mean_price,NULL)),MAX(IF(months_ago=12, mean_price,NULL)))*100,2) AS mean_growth_12m,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL))-MAX(IF(months_ago=60, mean_price,NULL)),MAX(IF(months_ago=60, mean_price,NULL)))*100,2) AS mean_growth_5yr,
  ROUND(SAFE_DIVIDE(MAX(IF(months_ago=0,mean_price,NULL))-MAX(IF(months_ago=120,mean_price,NULL)),MAX(IF(months_ago=120,mean_price,NULL)))*100,2) AS mean_growth_10yr,
  RANK() OVER (PARTITION BY anchor_type ORDER BY SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=3,  median_price,NULL)),MAX(IF(months_ago=3,  median_price,NULL))) DESC) AS rank_growth_3m,
  RANK() OVER (PARTITION BY anchor_type ORDER BY SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=6,  median_price,NULL)),MAX(IF(months_ago=6,  median_price,NULL))) DESC) AS rank_growth_6m,
  RANK() OVER (PARTITION BY anchor_type ORDER BY SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=12, median_price,NULL)),MAX(IF(months_ago=12, median_price,NULL))) DESC) AS rank_growth_12m,
  RANK() OVER (PARTITION BY anchor_type ORDER BY SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=60, median_price,NULL)),MAX(IF(months_ago=60, median_price,NULL))) DESC) AS rank_growth_5yr,
  RANK() OVER (PARTITION BY anchor_type ORDER BY SAFE_DIVIDE(MAX(IF(months_ago=0,median_price,NULL))-MAX(IF(months_ago=120,median_price,NULL)),MAX(IF(months_ago=120,median_price,NULL))) DESC) AS rank_growth_10yr,
  CURRENT_TIMESTAMP() AS refreshed_at
FROM base GROUP BY 1,2,3
''')

### 4j. Street summary

In [ ]:
run_query('street_summary', f'''
CREATE OR REPLACE TABLE {D}.street_summary AS
WITH latest AS (SELECT MAX(year_month) AS latest_ym FROM {D}.transactions_raw),
base AS (
  SELECT r.street,r.postcode_sector,r.local_authority,r.new_build,r.price,
    DATE_DIFF((SELECT PARSE_DATE('%Y-%m',latest_ym) FROM latest),
      PARSE_DATE('%Y-%m',r.year_month),MONTH) AS months_ago
  FROM {D}.transactions_raw r
  WHERE r.record_status!='D' AND r.street IS NOT NULL
)
SELECT street,postcode_sector,local_authority,
  (SELECT PARSE_DATE('%Y-%m',latest_ym) FROM latest) AS anchor_date,
  COUNTIF(new_build AND months_ago BETWEEN 0 AND 11)     AS new_build_count_12m,
  APPROX_QUANTILES(IF(new_build AND months_ago BETWEEN 0 AND 11,price,NULL),100)[OFFSET(50)] AS new_build_median_price,
  ROUND(AVG(IF(new_build AND months_ago BETWEEN 0 AND 11,price,NULL)),0)  AS new_build_mean_price,
  ROUND(SAFE_DIVIDE(APPROX_QUANTILES(IF(new_build AND months_ago BETWEEN 0 AND 11,price,NULL),100)[OFFSET(50)]-
    APPROX_QUANTILES(IF(new_build AND months_ago BETWEEN 12 AND 23,price,NULL),100)[OFFSET(50)],
    APPROX_QUANTILES(IF(new_build AND months_ago BETWEEN 12 AND 23,price,NULL),100)[OFFSET(50)])*100,2) AS new_build_growth_12m,
  COUNTIF(NOT new_build AND months_ago BETWEEN 0 AND 11) AS existing_count_12m,
  APPROX_QUANTILES(IF(NOT new_build AND months_ago BETWEEN 0 AND 11,price,NULL),100)[OFFSET(50)] AS existing_median_price,
  ROUND(AVG(IF(NOT new_build AND months_ago BETWEEN 0 AND 11,price,NULL)),0) AS existing_mean_price,
  ROUND(SAFE_DIVIDE(APPROX_QUANTILES(IF(NOT new_build AND months_ago BETWEEN 0 AND 11,price,NULL),100)[OFFSET(50)]-
    APPROX_QUANTILES(IF(NOT new_build AND months_ago BETWEEN 12 AND 23,price,NULL),100)[OFFSET(50)],
    APPROX_QUANTILES(IF(NOT new_build AND months_ago BETWEEN 12 AND 23,price,NULL),100)[OFFSET(50)])*100,2) AS existing_growth_12m,
  COUNTIF(months_ago BETWEEN 0 AND 11) AS total_count_12m,
  APPROX_QUANTILES(IF(months_ago BETWEEN 0 AND 11,price,NULL),100)[OFFSET(50)] AS total_median_price,
  ROUND(AVG(IF(months_ago BETWEEN 0 AND 11,price,NULL)),0) AS total_mean_price,
  CURRENT_TIMESTAMP() AS refreshed_at
FROM base GROUP BY 1,2,3,4
''')

## 5. Pipeline summary

In [ ]:
TABLES = [
    ('transactions_raw',              'ingested_at'),
    ('monthly_trends_sector',         'refreshed_at'),
    ('monthly_trends_sector_by_type', 'refreshed_at'),
    ('new_dev_trends_sector',         'refreshed_at'),
    ('sales_trends_sector',           'refreshed_at'),
    ('price_growth_sector',           'refreshed_at'),
    ('annual_price_growth_sector',    'refreshed_at'),
    ('cagr_sector_by_type',           'refreshed_at'),
    ('sector_percentile_bands',       'refreshed_at'),
    ('price_growth_local_authority',  'refreshed_at'),
    ('street_summary',                'refreshed_at'),
]
parts = ' UNION ALL '.join([
    f"SELECT '{t}' AS table_name, COUNT(*) AS row_count, MAX(CAST({ts} AS STRING)) AS last_updated FROM {D}.{t}"
    for t, ts in TABLES
])
summary = client.query(parts + ' ORDER BY table_name').to_dataframe()
print(f'=== Pipeline complete [{ENV.upper()}] — {latest_month.strftime("%B %Y")} ===')
print(f'Dataset   : {PROJECT}.{DATASET}')
print(f'First run : {FIRST_RUN}\n')
print(summary.to_string(index=False))

## 6. Promote dev -> prod
Run this cell manually after validating dev output. Set `PROMOTE = True` to execute.

**Only run when `ENV = 'dev'` and you are happy with all row counts above.**

In [ ]:
PROMOTE = False  # <-- set True to copy dev -> prod

if not PROMOTE:
    print('Promotion skipped. Set PROMOTE = True to copy dev -> prod.')
else:
    if ENV != 'dev':
        raise RuntimeError("Set ENV = 'dev' before promoting.")
    prod_dataset = DATASETS['prod']
    to_promote = [t for t, _ in TABLES] + ['transactions_staging']
    for table in to_promote:
        src = f'{PROJECT}.{DATASETS["dev"]}.{table}'
        dst = f'{PROJECT}.{prod_dataset}.{table}'
        print(f'  Copying {table}...', end=' ', flush=True)
        try:
            client.copy_table(src, dst, job_config=bigquery.CopyJobConfig(
                write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)).result()
            print('done')
        except Exception as e:
            print(f'SKIPPED ({e})')
    print(f'Promotion complete -> {PROJECT}.{prod_dataset}')